# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mocha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mocha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

c:\AIProjects\bootcamp\AIE7\07_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
c:\AIProjects\bootcamp\AIE7\07_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
c:\AIProjects\bootcamp\AIE7\07_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '8160e4'. Skipping!
Property 'summary' already exists in node '277184'. Skipping!
Property 'summary' already exists in node '07a9e6'. Skipping!
Property 'summary' already exists in node 'cb2e41'. Skipping!
Property 'summary' already exists in node '77a6c5'. Skipping!
Property 'summary' already exists in node '1db241'. Skipping!
Property 'summary' already exists in node '7a4a90'. Skipping!
Property 'summary' already exists in node '2320e9'. Skipping!
Property 'summary' already exists in node '36b0e1'. Skipping!
Property 'summary' already exists in node 'e344bc'. Skipping!
Property 'summary' already exists in node '64af10'. Skipping!
Property 'summary' already exists in node 'abeed0'. Skipping!
Property 'summary' already exists in node '6db4c7'. Skipping!
Property 'summary' already exists in node '17dbdc'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '8160e4'. Skipping!
Property 'summary_embedding' already exists in node '277184'. Skipping!
Property 'summary_embedding' already exists in node '36b0e1'. Skipping!
Property 'summary_embedding' already exists in node '07a9e6'. Skipping!
Property 'summary_embedding' already exists in node '2320e9'. Skipping!
Property 'summary_embedding' already exists in node '6db4c7'. Skipping!
Property 'summary_embedding' already exists in node '1db241'. Skipping!
Property 'summary_embedding' already exists in node '7a4a90'. Skipping!
Property 'summary_embedding' already exists in node '64af10'. Skipping!
Property 'summary_embedding' already exists in node 'abeed0'. Skipping!
Property 'summary_embedding' already exists in node 'cb2e41'. Skipping!
Property 'summary_embedding' already exists in node '77a6c5'. Skipping!
Property 'summary_embedding' already exists in node 'e344bc'. Skipping!
Property 'summary_embedding' already exists in node '17dbdc'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


SingleHopSpecificQuerySynthesizer: Generates straightforward, fact-based questions that can be answered by looking up a single piece of information in the documents. Example: “What is Volume 2?”

MultiHopAbstractQuerySynthesizer:
Creates more complex questions that require connecting information from multiple parts of the documents, often in a more general or conceptual way. These questions test the system’s ability to synthesize and reason across different sections. Example: “How do different academic year definitions for various programs affect compliance?”

MultiHopSpecificQuerySynthesizer:
Produces detailed, multi-step questions that require gathering and combining specific facts from several places in the documents. These are precise but require reasoning over multiple hops. Example: “How do Chapters 2 and 3 collectively address the requirements for disbursement timing?”


Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is Volume 2?,"[Chapter 1 Academic Years, Academic Calendars,...",The provided context does not include a defini...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding th...,[Regulatory Citations Academic year minimums: ...,Regulatory citations indicate that 34 CFR 668....,single_hop_specifc_query_synthesizer
2,What is included in Chapter 3 regarding clinic...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,What is Title IV and how it work in disbursement?,[Non-Term Characteristics A program that measu...,Title IV programs are subject to payment perio...,single_hop_specifc_query_synthesizer
4,What is Volume 7 about?,[both the credit or clock hours and the weeks ...,Volume 7 provides guidance on the disbursement...,single_hop_specifc_query_synthesizer
5,How does credit hour allocation for clinical e...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in a program de...,multi_hop_abstract_query_synthesizer
6,How do different academic year definitions for...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Different academic year definitions for variou...,multi_hop_abstract_query_synthesizer
7,How does policy compliance for term length and...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Policy compliance for term length and structur...,multi_hop_abstract_query_synthesizer
8,How do Chapters 2 and 3 relate to the requirem...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Chapter 2 details the requirements for definin...,multi_hop_specific_query_synthesizer
9,Volume 8 and Volume 7 both talk about disburse...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 explains that if a student accelerate...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '266c77'. Skipping!
Property 'summary' already exists in node '2a7fcc'. Skipping!
Property 'summary' already exists in node '6f6848'. Skipping!
Property 'summary' already exists in node 'da4656'. Skipping!
Property 'summary' already exists in node '595d80'. Skipping!
Property 'summary' already exists in node '5932ca'. Skipping!
Property 'summary' already exists in node '4247dc'. Skipping!
Property 'summary' already exists in node 'c582e8'. Skipping!
Property 'summary' already exists in node '0f1cf2'. Skipping!
Property 'summary' already exists in node '678378'. Skipping!
Property 'summary' already exists in node '19aa8c'. Skipping!
Property 'summary' already exists in node '0a0cd9'. Skipping!
Property 'summary' already exists in node '52924a'. Skipping!
Property 'summary' already exists in node '0790c5'. Skipping!


...

Property 'summary_embedding' already exists in node '266c77'. Skipping!
Property 'summary_embedding' already exists in node '4247dc'. Skipping!
Property 'summary_embedding' already exists in node 'da4656'. Skipping!
Property 'summary_embedding' already exists in node '6f6848'. Skipping!
Property 'summary_embedding' already exists in node '19aa8c'. Skipping!
Property 'summary_embedding' already exists in node '5932ca'. Skipping!
Property 'summary_embedding' already exists in node 'c582e8'. Skipping!
Property 'summary_embedding' already exists in node '0f1cf2'. Skipping!
Property 'summary_embedding' already exists in node '595d80'. Skipping!
Property 'summary_embedding' already exists in node '0790c5'. Skipping!
Property 'summary_embedding' already exists in node '52924a'. Skipping!
Property 'summary_embedding' already exists in node '2a7fcc'. Skipping!
Property 'summary_embedding' already exists in node '678378'. Skipping!
Property 'summary_embedding' already exists in node '0a0cd9'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is Title IV in relation to academic progr...,"[Chapter 1 Academic Years, Academic Calendars,...","For Title IV purposes, the academic year is de...",single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding ac...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(a) pertains to the minimum number...,single_hop_specifc_query_synthesizer
2,What is a standard term in educational programs?,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
3,Title IV what is it,[Non-Term Characteristics A program that measu...,Title IV programs are subject to specific regu...,single_hop_specifc_query_synthesizer
4,Considering the differences between standard a...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in standard ter...,multi_hop_abstract_query_synthesizer
5,how disbursement timing in subscription-based ...,[<1-hop>\n\nboth the credit or clock hours and...,"In the context of financial aid disbursement, ...",multi_hop_abstract_query_synthesizer
6,How does the timing of disbursement requiremen...,[<1-hop>\n\nboth the credit or clock hours and...,The disbursement timing requirements for finan...,multi_hop_abstract_query_synthesizer
7,what is the regualtions and compliance for aca...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulations and compliance for academic ye...,multi_hop_abstract_query_synthesizer
8,How do the disbursement timing rules in subscr...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,The disbursement timing rules in subscription-...,multi_hop_specific_query_synthesizer
9,so in chapter 3 how does disbursement timing w...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,"In chapter 3, disbursement timing in subscript...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct PLUS Loan or student Federal PLUS Loan  \n- Subsidized Federal Stafford Loans  \n- Unsubsidized Federal Stafford Loans  \n- Federal SLS Loans  \n- Federal PLUS Loans  \n- Direct Subsidized Loan  \n- Direct Unsubsidized Loan  \n- Direct Consolidation Loan  \n- Federal Consolidation Loan\n\nNote that Subsidized and Unsubsidized Federal Stafford Loans, Federal SLS Loans, and Federal PLUS Loans were made under the Federal Family Education Loan (FFEL) Program before the authority to make new FFEL Program loans ended effective July 1, 2010. New loans are primarily Direct Loans including Direct Subsidized Loans, Direct Unsubsidized Loans, and Direct PLUS Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:


##### ✅ Answer:

qa_evaluator:  Evaluates the factual correctness of the model’s answer by comparing it to the reference answer. It checks if the response is accurate and matches the expected information.

labeled_helpfulness_evaluator:
Assesses how helpful the answer is to the user, considering both the question and the reference answer. It measures whether the response provides useful, actionable, or supportive information.

empathy_evaluator:
Judges whether the response is empathetic—specifically, if it makes the user feel heard and understood. It looks for kindness and emotional resonance in the answer.


## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'notable-broccoli-44' at:
https://smith.langchain.com/o/a65ab79e-a9a4-510f-b5d1-9a57ec211031/datasets/012ba5c2-5227-4436-a0b6-96ad07896b6b/compare?selectedSessions=dd87f1b1-019a-464c-8697-18e7fd37254c




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Chapters 2 and 3 collectively address t...,I don't know.,None,Chapter 2 outlines the general requirements fo...,0,0,0,0.917887,2a1343cf-c8e0-4280-9be9-e3f0e56ade6e,edcf19ad-f83d-48d0-8efa-59f1c0606102
1,Whre do I find info in Appendix A and B about ...,I don't know.,None,"The detailed guidance on disbursement rules, i...",0,0,0,0.656262,67b84856-2636-4e83-bb14-57295864db43,29f0e5b6-0699-4192-88d7-b31c68a293d6
2,so in chapter 3 how does disbursement timing w...,I don't know.,None,"In chapter 3, disbursement timing in subscript...",0,0,0,1.032435,b3581f53-e093-41cf-894e-286564357102,84363315-c9ea-4e63-9507-db44498146ea
3,How do the disbursement timing rules in subscr...,I don't know.,None,The disbursement timing rules in subscription-...,0,0,0,1.523893,2fc84d39-9343-433c-b721-df8e36962ab3,f060c6cf-c147-4fca-8afb-c8e76763fc4d
4,what is the regualtions and compliance for aca...,The regulation 34 CFR 668.3(b) relates to the ...,None,The regulations and compliance for academic ye...,1,0,0,2.764453,e55a693d-c8d7-4596-b907-761e25183e49,70210437-abf3-4e71-9ee7-e663b1f1b0f8
5,How does the timing of disbursement requiremen...,Based on the provided context:\n\n**Subscripti...,None,The disbursement timing requirements for finan...,1,1,0,9.522738,7d3a29b9-4549-4bd2-8707-6637412620e8,90bdf01c-705c-475c-8039-efe591f62fe0
6,how disbursement timing in subscription-based ...,"Based on the provided context, the connection ...",None,"In the context of financial aid disbursement, ...",1,0,0,7.162190,0b4c5385-a527-448d-b917-3fad6f7eaf4b,90bf90b1-a1af-4992-97b3-431a6457d9bb
7,Considering the differences between standard a...,Based on the provided context:\n\n1. **Inclusi...,None,The inclusion of clinical work in standard ter...,1,1,0,11.666972,5f077ad4-ee39-49fe-8587-f53299ce26d6,0a660329-7c30-4ece-98f3-ba1ff632b8d4
8,Title IV what is it,"Based on the provided context, Title IV refers...",None,Title IV programs are subject to specific regu...,1,1,0,2.034174,367db68b-fd64-4fec-9ded-f36862b69078,a93b65f2-3b42-46d2-9da7-3e65fedebd31
9,What is a standard term in educational programs?,A standard term in educational programs is a d...,None,Inclusion of Clinical Work in a Standard Term ...,0,1,0,4.201579,8b60508c-bdbf-45ee-9859-2edf015db830,e668f4c3-8a16-4acb-b7b9-f7905b91d824


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

Context Coverage: Larger chunks provide more context per retrieval, which can help answer complex questions that require broader information.
Relevance vs. Noise: Smaller chunks are more focused and relevant, but may miss important context; larger chunks may include irrelevant information, diluting answer quality.
Retrieval Precision: Smaller chunks allow for more precise retrieval of relevant information, while larger chunks may make it harder to pinpoint specific answers.
Trade-off: There is a trade-off between having enough context (larger chunks) and maintaining high relevance (smaller chunks).
Impact on Results: The chosen chunk size directly influences the accuracy, helpfulness, and overall performance of the RAG syste

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Modifying the embedding model changes how your application represents and understands the meaning of text. More advanced or better-suited embedding models can capture semantic relationships more accurately, leading to improved retrieval of relevant information. Conversely, less capable models may miss important context or retrieve less relevant chunks, reducing answer quality. The choice of embedding model directly impacts the accuracy, relevance, and overall effectiveness of your RAG system. Selecting the right model is crucial for optimal performance.

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided, there are several types of loans available:\n\n1. **Direct Subsidized Loans**: These are loans where the amount you can borrow depends on your financial need (calculated as your Cost of Attendance minus other financial aid). They have an annual maximum limit and interest is subsidized by the government while you are in school.\n\n2. **Direct Unsubsidized Loans**: These loans are available regardless of financial need, and you can qualify for additional unsubsidized loan amounts if you are independent or if your parent is ineligible for a Direct PLUS Loan. Interest accrues while you are in school.\n\n3. **Direct PLUS Loans**: These are loans that parents of dependent students can take out to help pay for the student’s Cost of Attendance, assuming eligibility. There’s no fixed loan limit for PLUS Loans, but they cannot exceed the Cost of Attendance minus other aid.\n\nIt’s also noted that if a dependent student’s parent can

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'warm-wash-19' at:
https://smith.langchain.com/o/a65ab79e-a9a4-510f-b5d1-9a57ec211031/datasets/012ba5c2-5227-4436-a0b6-96ad07896b6b/compare?selectedSessions=5d5542cd-c110-4e93-95d1-40b311bf6623




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Chapters 2 and 3 collectively address t...,Thank you for your thoughtful question. Based ...,None,Chapter 2 outlines the general requirements fo...,0,0,1,5.720070,2a1343cf-c8e0-4280-9be9-e3f0e56ade6e,22a6fa0d-b9d1-4023-bc4c-6b8381a8d747
1,Whre do I find info in Appendix A and B about ...,Thank you for your question — I understand how...,None,"The detailed guidance on disbursement rules, i...",0,0,1,3.865045,67b84856-2636-4e83-bb14-57295864db43,d8f6b3e0-357c-44be-b316-b8a7b599c69f
2,so in chapter 3 how does disbursement timing w...,Thank you for your thoughtful question about d...,None,"In chapter 3, disbursement timing in subscript...",0,0,1,4.779154,b3581f53-e093-41cf-894e-286564357102,42312ee4-6c99-42ea-8b8a-4b2ca323f796
3,How do the disbursement timing rules in subscr...,Thank you for your thoughtful question. Based ...,None,The disbursement timing rules in subscription-...,1,1,1,4.608508,2fc84d39-9343-433c-b721-df8e36962ab3,166f7454-bd72-46df-8fee-6506693d0bc6
4,what is the regualtions and compliance for aca...,Thank you for your question. Based on the info...,None,The regulations and compliance for academic ye...,0,0,1,2.321598,e55a693d-c8d7-4596-b907-761e25183e49,610dd7df-0809-48eb-9f52-7b8ab7fe9540
5,How does the timing of disbursement requiremen...,Thank you for your thoughtful question about t...,None,The disbursement timing requirements for finan...,1,1,1,8.176819,7d3a29b9-4549-4bd2-8707-6637412620e8,3686862a-3deb-438b-8482-098b186c2aa8
6,how disbursement timing in subscription-based ...,Thank you for your thoughtful question about t...,None,"In the context of financial aid disbursement, ...",1,1,1,7.718805,0b4c5385-a527-448d-b917-3fad6f7eaf4b,bd1e37f2-579f-4441-ad90-29fee6687c05
7,Considering the differences between standard a...,Thank you for your thoughtful question. I can ...,None,The inclusion of clinical work in standard ter...,1,1,1,10.526607,5f077ad4-ee39-49fe-8587-f53299ce26d6,e710bba0-827d-4a73-b248-4d4cd48140b9
8,Title IV what is it,I see you're asking about Title IV. Based on t...,None,Title IV programs are subject to specific regu...,1,1,1,2.482170,367db68b-fd64-4fec-9ded-f36862b69078,1d0d748e-2439-48e4-8539-7eb7dd6dc21a
9,What is a standard term in educational programs?,Thank you for your thoughtful question! Based ...,None,Inclusion of Clinical Work in a Standard Term ...,0,0,1,3.518922,8b60508c-bdbf-45ee-9859-2edf015db830,36e5738d-125d-462e-8107-9e194d16e1f1


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

##### ✅ Answer:

Screenshots: 

Original: https://drive.google.com/file/d/1BUKblI39cjgbSWtcedsH7_ThPo4qA83v/view?usp=sharing
Edited: https://drive.google.com/file/d/1TkvhaCWCskQjlUeaR2hYmAAv1b7sBuew/view?usp=sharing


Explanation of Metric Changes Between the Two Chains
After modifying the RAG chain (by increasing chunk size, switching to a larger embedding model, and adding an empathy-focused prompt), the following changes were observed:
Correctness decreased from 0.58 to 0.50.
Helpfulness decreased from 0.50 to 0.416.
Empathy increased from 0 to 1.0.
These changes can be explained by the specific modifications made:
1. Empathy Prompt Addition: The new prompt explicitly instructed the model to answer with empathy and kindness, ensuring the user feels heard. This directly improved the empathy score, as the model’s responses were now evaluated as empathetic by the evaluator.
2. Correctness Decrease: The focus on empathetic language, combined with a larger chunk size, may have led the model to generate responses that were more conversational or supportive but less strictly factual. Larger chunks can also introduce more irrelevant information, making it harder for the model to extract and present the precise answer, which likely contributed to the drop in correctness.
3. Helpfulness Decrease: Helpfulness is influenced by both factual accuracy and user support. While the model became more empathetic, the decrease in correctness meant that answers were less directly useful or actionable, causing the helpfulness score to drop as well.
4. Embedding Model Change: Although a larger embedding model was used (from text-embedding-3-small to text-embedding-3-large), the benefits may have been offset by the increased chunk size and the new prompt focus, resulting in no improvement—and even a decline—in factual retrieval and answer quality.